# Estimate an activation energy from reaction-rate data

Fit a temperature-dependent first-order rate constant, estimate its molar
activation energy, and predict a rate and half-life inside the observed interval.
Run the setup in [examples/README.md](README.md) first.

The [IUPAC Arrhenius equation](https://goldbook.iupac.org/terms/view/A00446) is
$$
k(T)=A\exp\!\left(-\frac{E_a}{RT}\right).
$$
Here $T$ is absolute temperature, $R$ is the molar gas constant, $E_a$ is the
molar activation energy, and $A$ has the same units as the rate constant $k$.
We assume $E_a$ and $A$ remain constant over this temperature interval.

These are **synthetic data**, not measurements of a named reaction. They were
generated with $E_a=52\,\mathrm{kJ/mol}$ and $A=2\times10^7\,\mathrm{s^{-1}}$,
with fixed log-rate perturbations $[0.020,-0.025,0.010,-0.015,0.025,-0.015]$,
then rounded. The known generating parameters provide a numerical check;
they do not validate the model for a real reaction.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from unitx import math
from unitx.si import SI, si

# Recorded Celsius temperatures are points, not temperature differences.
temperature_readings = np.asarray([15, 25, 35, 45, 55, 65]) @ si.degC
temperature = temperature_readings.to_quantity(si.K)
rate = np.asarray([0.458888, 0.908486, 1.858462, 3.430419, 6.499562, 10.972105]) / si.min
reference_rate = 1 / si.min

## Fit at an explicit numerical boundary

Normalize the rate before taking its logarithm:
$$
x=\frac{1}{T},\qquad y=\ln\!\left(\frac{k}{k_{\rm ref}}\right),\qquad
 y=mx+b.
$$
Thus $m=-E_a/R$ has unit K, $b=\ln(A/k_{\rm ref})$ is dimensionless, and
$$
E_a=-mR,\qquad A=k_{\rm ref}\exp(b).
$$

The temperatures have already been converted to K, so `(1 / temperature).value`
contains inverse temperatures in K$^{-1}$. `np.polyfit` receives these numerical
values and the dimensionless log rates. Attach K to its fitted slope before
continuing the physical calculation. Keep the fitted scalars in NumPy's
floating-point domain; unitx's NumPy adapter evaluates catalogue constants in
that domain without changing `SI.R`.

In [ ]:
log_rate = math.log(rate / reference_rate)
coefficients, covariance = np.polyfit(
    (1 / temperature).value,
    log_rate.value,
    deg=1,
    cov=True,
)
slope = np.float64(coefficients[0]) * si.K
slope_standard_error = np.sqrt(np.float64(covariance[0, 0])) * slope.unit
activation_energy = (-slope * SI.R).to('kJ/mol')
energy_standard_error = (slope_standard_error * SI.R).to(activation_energy.unit)
prefactor = (reference_rate * math.exp(np.float64(coefficients[1]))).to('s^-1')

print(f'Activation energy: {activation_energy:.3f}')
print(f'Standard error from residual scatter: {energy_standard_error:.3f}')
print(f'Prefactor: {prefactor:.4g}')

## Interpret and plot the fit

Ordinary least squares here assumes negligible temperature error and independent
log-rate errors with a common variance. The covariance returned by `polyfit`
estimates that variance from residual scatter, using $6-2=4$ residual degrees
of freedom. Therefore $R\sqrt{\operatorname{Cov}_{00}}$ is a model-based standard
error for $E_a$, not a confidence interval or a measure of systematic error.
The estimates of $A$ and $E_a$ are correlated; no prediction uncertainty band is
computed here.

Expect an activation energy near $51.9\,\mathrm{kJ/mol}$ with standard error
near $0.45\,\mathrm{kJ/mol}$. Plot a dimensionless inverse-temperature coordinate,
$1000\,\mathrm{K}/T$. The label is derived from the same scale used in the plot.
The curve spans only the observed temperature range.

In [ ]:
curve_temperature = (
    np.linspace(temperature.value.min(), temperature.value.max(), 200, dtype=np.float64) * si.K
)
curve_rate = prefactor * math.exp(-activation_energy / (SI.R * curve_temperature))
temperature_scale = 1000 * si.K
plot_temperature = temperature_scale / temperature
curve_coordinate = temperature_scale / curve_temperature

fig, ax = plt.subplots(figsize=(6, 4), layout='constrained')
ax.scatter(plot_temperature.value, log_rate.value, label='Synthetic observations')
ax.plot(
    curve_coordinate.value,
    math.log(curve_rate / reference_rate).value,
    label='Arrhenius fit',
)
ax.set(
    xlabel=f'{temperature_scale} / T',
    ylabel='ln(k / k_ref)',
    title='Activation-energy fit',
)
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()

## Predict inside the observed interval

At a fixed temperature, a first-order decay obeys $c(t)=c(0)\exp(-kt)$, hence
$$
t_{1/2}=\frac{\ln 2}{k}.
$$
Use the fitted parameters to predict at $40\,{}^\circ\mathrm C$, inside the
15--65 $^\circ\mathrm C$ data interval. Convert the temperature point to a Kelvin
quantity before evaluating the exponent. This is a prediction under the stated
Arrhenius and first-order models, with no allowance for model error.

In [ ]:
prediction_reading = np.float64(40) @ si.degC
prediction_temperature = prediction_reading.to_quantity(si.K)
predicted_rate = prefactor * math.exp(-activation_energy / (SI.R * prediction_temperature))
half_life = (math.log(2) / predicted_rate).to(si.s)

print(f'Predicted rate at {prediction_reading}: {predicted_rate.to(rate.unit):.4f}')
print(f'First-order half-life: {half_life:.2f}')

## Numerical checks

Compare the estimates with the known generating parameters. Also check that
converting the input rate storage unit does not alter the normalized fit data,
and that the reconstructed physical model agrees with the fitted straight line.
The half-life identity checks the prediction's time/rate units together.

In [ ]:
assert activation_energy.isclose(52 * si.kJ / si.mol, rel_tol=0.02)
assert prefactor.isclose(2e7 / si.s, rel_tol=0.1)
assert (predicted_rate * half_life).isclose(math.log(2), rel_tol=1e-12)
np.testing.assert_allclose(
    math.log(rate.to('s^-1') / reference_rate).value,
    log_rate.value,
    rtol=1e-12,
    atol=1e-12,
)
fitted_rate = prefactor * math.exp(-activation_energy / (SI.R * temperature))
np.testing.assert_allclose(
    math.log(fitted_rate / reference_rate).value,
    coefficients[0] * (1 / temperature).value + coefficients[1],
    rtol=1e-12,
    atol=1e-12,
)